In [1]:
from B0_MODEL import UNET
from B2_UTILS import (load_checkpoint, save_checkpoint, get_loaders,
                      check_accuracy, save_predictions_as_imgs)
import torch
from tqdm import tqdm  # makes your loops show a smart progress meter
import torch.nn as nn
import torch.optim as optim
import os
import json

In [2]:
# Set training epoch munbers
TRAIN = True
training_epoch_num = 1000

# Test training?
BATCH_1_TEST = True


In [3]:
# ====================== Hyperparameters etc. ========================
MAIN_PATH = '/home/qiboxu/Database/U_NET/EMDB_PDB_for_U_Net/Filtered_Dateset/Training/TEST'
MODEL_TYPES = ['secondary_strctures', 'key_atoms', 'residue_types']
MAP_DIRS_TYPES = [['map_sample'], ['map_sample', 'secondary_strctures'],
                  ['map_sample', 'secondary_strctures', 'key_atoms']]

type_index = 0
model_type = MODEL_TYPES[type_index]
map_dirs = MAP_DIRS_TYPES[type_index]

In [4]:
# ====================== Hyperparameters etc. ========================
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
batch_size = 8  # batch_size vs GPU Memory: 1vs2877; 2vs3485; 4vs4668; 8vs7162

num_workers = 2
learning_rate = 1e-4
pin_memory = True
IMAGE_HEIGHT, IMAGE_WIDTH, IMAGE_DEPTH = [64, 64, 64]

# Set input data
data_path = os.path.join(MAIN_PATH, 'train_val_data')
train_map_dirs = [
    os.path.join(data_path, 'train', map_dir) for map_dir in map_dirs
]
val_map_dirs = [
    os.path.join(data_path, 'val', map_dir) for map_dir in map_dirs
]
train_model_dir = os.path.join(data_path, 'train', model_type)
val_model_dir = os.path.join(data_path, 'val', model_type)

# Set for loading model weights
LOAD_MODEL = False
load_epoch_num = 0
load_model_dir = os.path.join(data_path, 'checkpoint')
load_model_name = os.path.join(data_path, 'checkpoint',
                               f'{model_type}.epoch{load_epoch_num}.pth.tar')
evl_folder_name = os.path.join(data_path, 'evaluation', model_type)

# Load the class weights of all model types
weight_path = os.path.join(data_path, 'class_weight_for_training.txt')
with open(weight_path, "r") as file:
    weight_all = json.load(file)
class_weight = weight_all[model_type]
CLASS_WEIGHTS = torch.FloatTensor(class_weight).to(device=DEVICE)

# Set model channels
in_channels = len(map_dirs)
out_channels = len(class_weight)

In [5]:
def train_fn(loader, model, optimizer, loss_fn, scaler):
    loop = tqdm(loader)
    if BATCH_1_TEST:
        data, targets = next(iter(loader))
        for epoch in range(1000):
            data = data.to(device=DEVICE)
            targets = targets.long().to(device=DEVICE)
            # forward
            with torch.cuda.amp.autocast():
                predictions = model(data)
                loss = loss_fn(predictions, targets)
            # backward
            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            # update tqdm loop
            loop.set_postfix(loss=loss.item())

    else:
        for batch_idx, (data, targets) in enumerate(loop):
            data = data.to(device=DEVICE)
            targets = targets.long().to(device=DEVICE)
            # forward
            with torch.cuda.amp.autocast():
                predictions = model(data)
                loss = loss_fn(predictions, targets)
            # backward
            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            # update tqdm loop
            loop.set_postfix(loss=loss.item())

In [6]:
model = UNET(in_channels=in_channels, out_channels=out_channels).to(DEVICE)

In [7]:
loss_fn = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS)
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

train_loader, val_loader = get_loaders(train_map_dirs=train_map_dirs,
                                        train_model_dir=train_model_dir,
                                        val_map_dirs=val_map_dirs,
                                        val_model_dir=val_model_dir,
                                        batch_size=batch_size,
                                        num_workers=num_workers,
                                        pin_memory=pin_memory)

In [8]:
scaler = torch.cuda.amp.GradScaler()

In [9]:
train_fn(train_loader, model, optimizer, loss_fn, scaler)

  0%|          | 0/1 [00:00<?, ?it/s]/home/qiboxu/Software/anaconda3/envs/torch2.0.1-py3.11/lib/python3.11/site-packages/torch/nn/modules/conv.py:608: UserWarning: Applied workaround for CuDNN issue, install nvrtc.so (Triggered internally at /opt/conda/conda-bld/pytorch_1682343995622/work/aten/src/ATen/native/cudnn/Conv_v8.cpp:80.)
  return F.conv3d(
/opt/conda/conda-bld/pytorch_1682343995622/work/aten/src/ATen/native/cuda/NLLLoss2d.cu:103: nll_loss2d_forward_kernel: block: [1,0,0], thread: [229,0,0] Assertion `t >= 0 && t < n_classes` failed.
/opt/conda/conda-bld/pytorch_1682343995622/work/aten/src/ATen/native/cuda/NLLLoss2d.cu:103: nll_loss2d_forward_kernel: block: [1,0,0], thread: [230,0,0] Assertion `t >= 0 && t < n_classes` failed.
/opt/conda/conda-bld/pytorch_1682343995622/work/aten/src/ATen/native/cuda/NLLLoss2d.cu:103: nll_loss2d_forward_kernel: block: [1,0,0], thread: [803,0,0] Assertion `t >= 0 && t < n_classes` failed.
/opt/conda/conda-bld/pytorch_1682343995622/work/aten/src

RuntimeError: GET was unable to find an engine to execute this computation